In [1]:
# import necessary libraries
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec

/mnt/e/TanishProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import re

In [3]:
# load environment variables
load_dotenv()

True

In [4]:
# initialize LLM 
model = init_chat_model("google_genai:gemini-2.5-flash")

In [5]:
# load the required document
file_path = "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx"
loader = UnstructuredWordDocumentLoader(
            file_path,
            mode="elements"                             # breaks document into structured chunks based on layout
        )
docs = loader.load()

In [6]:
def is_header_footer(doc):
    text = doc.page_content.strip()
    category = doc.metadata.get("category", "")
    
    NOISE_PATTERNS = [
        r"Version\s+\d+[\.\d]*\s*\|",                       # "Version 5.2 |"
        r"Internal Restricted",                             # footer 
        r"Not for External Distribution",                   # footer 
        r"©\s*20\d{2}\s",                                   # copyright footer
        r"Enterprise Policy & Procedures Handbook",         # header 
        r"CONFIDENTIAL",                                    # header 
        r"NexaCorp Global, Inc."                            # cover page
    ]
    return any(re.search(p, text, re.IGNORECASE) for p in NOISE_PATTERNS)


# Apply filters 
docs_clean = []

for doc in docs:
    if is_header_footer(doc):
        continue

    docs_clean.append(doc)
    
print(f"Before: {len(docs)}")
print(f"After: {len(docs_clean)}")

Before: 530
After: 510


In [7]:
print(docs[10])

page_content='Tel: +1 (415) 700-9000  |  hr@nexacorp.com  |  www.nexacorp.com' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'category_depth': 0, 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'last_modified': '2026-04-30T05:28:12', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'parent_id': '455472029b5887c8acdc6c539fd7b95a', 'category': 'UncategorizedText', 'element_id': '96a4b1466765e57ee816307d4291b752'}


In [8]:
print(docs_clean[10])

page_content='' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'languages': ['eng'], 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'PageBreak', 'element_id': '374c9b677aede328ee9f4fdefb686eb5'}


In [9]:
structured_docs = []
current_section = "General"                             # default section name
current_part = "General"

for doc in docs_clean:
    text = doc.page_content.strip()                     # page content without extra spaces
    category = doc.metadata.get("category", "")         # to get type of element like Title, Text etc

    if not text:                                        # skip empty chunks
        continue

    if category == "Title":                             # if chunk is Title update the current_section
        if re.search(r"Part\s+[IVX]+", text):
            current_part = text
        else:
            current_section = text
        continue 
    
    structured_docs.append({                            # add section and content to structured_docs
        "part": current_part,
        "section": current_section,
        "content": text
    })

In [10]:
# text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      
    chunk_overlap=100,
    separators=["\n\n", "\n", "."]  
)

final_chunks = []

for item in structured_docs:
    section = item["section"]                           # extract the section and text of merged doc
    text = item["content"]

    chunks = splitter.split_text(text)                  # split text into chunks

    for chunk in chunks:
        if len(chunk.split()) < 40:                     # to filter out small chunks that can be noise 
            continue
        final_chunks.append({
            "content": f"{item['part']} > {item['section']}\n{chunk}",
            "part": item["part"],
            "section": item["section"]
        })

In [11]:
print(len(final_chunks))

111


In [12]:
# convert chunks to LangChain Documents
documents = []                                      

for chunk in final_chunks:
    documents.append(
        Document(
            page_content=chunk["content"],
            metadata={
                "part": chunk["part"],
                "section": chunk["section"]
            }
        )
    )

In [13]:
print(len(documents))

111


In [14]:
# create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1736.46it/s]


In [15]:
from langchain_community.vectorstores import FAISS

In [16]:
# create Pinecone vector store
vectorstore = FAISS.from_documents(
        documents=documents,
        embedding=embeddings
    )

In [17]:
# create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",                        # reduces redundancy
    search_kwargs={"k": 3}
)

In [18]:
query = "Summarize Part I of the handbook"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\nResult {i+1}:")
    print(doc.page_content)
    print(doc.metadata)


Result 1:
General > Appendix F: Country-Specific Addenda Index
The following country-specific addenda supplement this handbook in the jurisdictions listed. Where an addendum exists, employees and managers in that country should consult both the main handbook and the relevant addendum. Addenda are available on the HR Portal under 'Country Policies'.
{'part': 'General', 'section': 'Appendix F: Country-Specific Addenda Index'}

Result 2:
General > Appendix A: Policy Revision History
Version Date Summary of Changes Approved By 1.0 Jan 2010 Initial policy handbook issued CEO, CLO 2.0 Jan 2013 Added social media and BYOD policies; expanded ethics section ELT 2.5 Jul 2014 LTI programme added; salary bands updated CPO, CFO 3.0 Jan 2017 GDPR readiness updates; whistleblower channel added; EEO expanded ELT 4.0 Jan 2019 Remote work framework introduced; cyber incident response added ELT 4.5 Jul 2020 COVID-19 response addendum; mental health benefits added CPO, CMO 5.0 Jan 2022 Complete rewrite; 

In [ ]:
for doc in documents[:5]:
    print(doc.metadata, "\n")
    print(doc.page_content[:800])